# Reference Resolution Dev Notebook

In [1]:
import random
import torch
import io
import pyarrow as pa
import os
import copy
import pytorch_lightning as pl
from sacred import Experiment
from PIL import Image
from tqdm import tqdm
import numpy as np
import skimage.io as skio
import matplotlib.pyplot as plt
from refer import REFER

from torch.optim import AdamW

from transformers import ElectraTokenizer

from refcoco_utils import get_bounded_subimage
from refcoco_utils import _config
from refcoco_utils import _loss_names

from meter.transforms import keys_to_transforms
from meter.config import ex
from meter.modules import METERTransformerSS
from meter.datamodules.multitask_datamodule import MTDataModule
from meter.datasets.base_dataset import BaseDataset

## RefCOCO Data

### Utility Functions

In [2]:
def get_sent_ids(refer):
    sent_ids = []
    for ref_id in refer.getRefIds():
        ref = refer.Refs[ref_id]
        for sent_id in ref['sent_ids']:
            sent_ids.append(sent_id)
    return sent_ids

### Import  Data

In [3]:
data_root = '/home/claytonfields/nlp/code/data/coco'  # contains refclef, refcoco, refcoco+, refcocog and images
dataset = 'refcoco' 
splitBy = 'unc'
refer = REFER(data_root, dataset, splitBy)

loading dataset refcoco into memory...
testing
creating index...
index created.
DONE (t=11.98s)


In [4]:
refer.IMAGE_DIR = '/home/claytonfields/nlp/code/data/coco/images/mscoco/train2014'

### Find Max Number of Objects

In [5]:
train_ids = refer.getRefIds()
size = []
for ref_id in train_ids:
    ref = refer.Refs[ref_id]
    img_id = ref['image_id']
    ann_id = ref['ann_id']
    objs = refer.imgToAnns[img_id]
    size.append(len(objs))
num_refs = len(train_ids)
max_size = np.max(size)
avg_size = np.mean(size)
median_size = np.median(size)
p_70 =  np.percentile(size, 70)
p_80 =  np.percentile(size, 80)
p_90 =  np.percentile(size, 90)
p_95 =  np.percentile(size, 95)
p_99 =  np.percentile(size, 99)
num_over_42 = np.sum(np.array(size) >= 42)

print(f'The most objects in any reference is {max_size}')
print(f'The average number of objects in each reference is {avg_size}')
print(f'The median number of objects in each reference is {median_size}')
print()
print(f'The 70th percentile of the number of objects in each reference is {p_70}')
print(f'The 80th percentile of the number of objects in each reference is {p_80}')
print(f'The 80th percentile of the number of objects in each reference is {p_90}')
print(f'The 95th percentile of the number of objects in each reference is {p_95}')
print(f'The 99th percentile of the number of objects in each reference is {p_99}')
print()
print(f'A max_bb of 42 would exclude {num_over_42} of {num_refs} refs')

The most objects in any reference is 75
The average number of objects in each reference is 10.60916
The median number of objects in each reference is 8.0

The 70th percentile of the number of objects in each reference is 13.0
The 80th percentile of the number of objects in each reference is 16.0
The 80th percentile of the number of objects in each reference is 21.0
The 95th percentile of the number of objects in each reference is 26.0
The 99th percentile of the number of objects in each reference is 41.0

A max_bb of 42 would exclude 453 of 50000 refs


### Find Max Number of Sentences

In [6]:
size = []
for ref_id in train_ids:
    ref = refer.Refs[ref_id]
    objs = refer.imgToAnns[img_id]
    size.append(len(ref['sentences']))
max_size = max(size)
print(f'The most sentences in any reference is {max_size}')

The most sentences in any reference is 6


## METER Model

In [7]:
_config = copy.deepcopy(_config)
pl.seed_everything(_config["seed"])
# dm = MTDataModule(_config, dist=False) 
model = METERTransformerSS(_config)

Global seed set to 0
Some weights of the model checkpoint at google/electra-small-discriminator were not used when initializing ElectraModel: ['discriminator_predictions.dense.weight', 'discriminator_predictions.dense_prediction.bias', 'discriminator_predictions.dense_prediction.weight', 'discriminator_predictions.dense.bias']
- This IS expected if you are initializing ElectraModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing ElectraModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


## Data Class for Ref Res with multiple samples

1. Adjust ref_res_classifier dimension to reflect max_bb

2.  May require padding to some max_num_bb.

The max number of objects in any ref in the training set is 75. Should consider some smaller number like 42. 42 represents the 99th percentile
    
**Below is a previous dataset class for reference**

**New class that produces uniform items padded to max_bb**

In [8]:
class RefcocoDataset(torch.utils.data.Dataset):

    def __init__(self, refer, tokenizer, split='', max_bb = 42):
        self.tokenizer = tokenizer
        self.refer = refer
        self.max_bb = max_bb
        self.split = split
        self.sent_ids = self.get_sent_ids()
        self.duds = []

    def __len__(self):
        return len(self.sent_ids)
    
    def get_sent_ids(self):
        sent_ids = []
        for ref_id in self.refer.getRefIds(split=self.split):
            ref = self.refer.Refs[ref_id]
            img_id = ref['image_id']
            objs = refer.imgToAnns[img_id]
            if len(objs) <= self.max_bb:
                for sent_id in ref['sent_ids']:
                    sent_ids.append(sent_id)
        return sent_ids
    
    def __getitem__(self, index):
        max_bb = self.max_bb
        sent = refer.Sents[index]
        ref = refer.sentToRef[index]
        img_id = ref['image_id']
        ann_id = ref['ann_id']
        objs = refer.imgToAnns[img_id]
        obj_ids = [obj['id'] for obj in objs]
        obj_pad = [0 for _ in range(max_bb-len(obj_ids))]
        obj_ids_total = obj_ids+obj_pad

        sub_images = []
        for obj in objs:
            x_a = get_bounded_subimage(refer, img_id, obj['id'], xs=224,ys=224, show=False)
            if x_a is not None:
                sub_images.append(x_a)
        
        num_sub_images = len(sub_images)
        num_pad = max_bb - num_sub_images 
        
        pad_image = torch.zeros(1,3,224,224)
        for _ in range(max_bb - num_sub_images):
            sub_images.append(pad_image)
        
        # text ids
        ids = tokenizer.encode(
            sent['sent'],
            padding="max_length",
            truncation=True,
            max_length=40,
            return_special_tokens_mask=True,
        )
        repeat_ids = torch.tensor(ids).repeat(num_sub_images,1)
        pad_ids =  torch.zeros(num_pad,40)
        text_ids = torch.cat((repeat_ids, pad_ids)).to(torch.long)
        # text masks
        num_tokens = torch.where(text_ids[0] > 0)[0].size(dim=0)
        masks = torch.cat((torch.ones(num_tokens), torch.zeros(40-num_tokens))).to(torch.long)
        repeat_masks = masks.repeat(num_sub_images,1)
        pad_masks = torch.zeros(num_pad, 40)
        text_masks = torch.cat((repeat_masks, pad_masks)).to(torch.long)
        # text_labels
        labels = torch.full((40,),-100)
        repeat_labels = labels.repeat(num_sub_images, 1)
        pad_labels = torch.zeros(num_pad, 40)
        text_labels = torch.cat((repeat_labels, pad_labels)).to(torch.long)

        return_dict = {
            'ann_id' : ann_id,
            'image' : [torch.cat(sub_images)],
            'obj_ids' : torch.tensor(obj_ids_total),
            'text' : sent['sent'],
            'text_ids' : text_ids,
            'text_labels' : text_labels,
            'text_masks' : text_masks
        }
        
        
            
        
        return return_dict

## Perform Ref Res with METER

**Define key variables and parameters**

In [9]:
optim = AdamW(model.parameters(), lr=1e-4)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

# Ref Res with METER
tokenizer = ElectraTokenizer.from_pretrained('google/electra-small-discriminator')
BATCH_SIZE = 1


epochs = 1
# loader = dm.train_dataloader()
optim = AdamW(model.parameters(), lr=1e-4)
loss_fn = torch.nn.functional.cross_entropy
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [10]:
ds = RefcocoDataset(refer, tokenizer, split='train', max_bb=42)
# ds = NewRefcocoDataset(refer, tokenizer)
train_params = {'batch_size': BATCH_SIZE,
                'shuffle': False,
                'num_workers': 0
                }

training_loader = torch.utils.data.DataLoader(ds, **train_params)

In [11]:
infer = model.infer(ds[0])
infer

{'text_feats': tensor([[[ 1.3259e-01,  4.9219e-03, -4.9616e-02,  ..., -6.4748e-02,
            8.6656e-02, -9.3155e-03],
          [ 5.9060e-01, -6.2583e-02, -7.0244e-01,  ...,  8.1228e-02,
            1.1158e-03,  2.1270e-02],
          [-1.3168e-02,  6.2440e-02, -4.8214e-01,  ...,  1.2478e-01,
            5.0424e-01,  7.6685e-01],
          ...,
          [ 2.6586e-01, -6.5517e-02, -2.8793e-02,  ...,  1.8412e-01,
           -3.4901e-04,  3.8993e-02],
          [ 4.1058e-01,  4.6175e-01,  3.0911e-01,  ..., -4.7214e-01,
            1.2931e-01,  2.2373e-01],
          [ 1.1499e-01,  1.2651e-01,  3.8770e-02,  ..., -3.5306e-02,
           -4.0765e-02,  3.5942e-01]],
 
         [[ 9.0919e-02,  4.4625e-03, -2.4278e-02,  ..., -4.4125e-02,
            1.0334e-01,  1.6819e-02],
          [ 5.7956e-01, -1.0934e-01, -6.4047e-01,  ...,  1.3411e-01,
            1.2219e-02, -7.4908e-02],
          [ 3.4540e-02,  2.1911e-01, -2.9422e-01,  ...,  4.7443e-01,
            4.4152e-01,  6.4030e-01],
     

In [13]:
batch = []
for i, data in enumerate(ds):
    if i==10:
        break
    batch.append(data)
batch

[{'ann_id': 1719310,
  'image': [tensor([[[[0.0549, 0.0549, 0.0549,  ..., 0.0078, 0.0000, 0.0000],
             [0.0667, 0.0667, 0.0627,  ..., 0.0118, 0.0118, 0.0118],
             [0.0627, 0.0627, 0.0627,  ..., 0.0039, 0.0039, 0.0039],
             ...,
             [0.2824, 0.2588, 0.2157,  ..., 0.6784, 0.7176, 0.7412],
             [0.2745, 0.2549, 0.2235,  ..., 0.8588, 0.8627, 0.8627],
             [0.2745, 0.2588, 0.2353,  ..., 0.6941, 0.6745, 0.6627]],
   
            [[0.0510, 0.0510, 0.0510,  ..., 0.0118, 0.0078, 0.0039],
             [0.0667, 0.0667, 0.0627,  ..., 0.0118, 0.0118, 0.0118],
             [0.0627, 0.0627, 0.0627,  ..., 0.0039, 0.0000, 0.0000],
             ...,
             [0.4314, 0.3922, 0.3216,  ..., 0.7804, 0.8196, 0.8471],
             [0.4157, 0.3843, 0.3255,  ..., 0.8863, 0.8706, 0.8627],
             [0.4157, 0.3882, 0.3373,  ..., 0.6471, 0.6078, 0.5843]],
   
            [[0.0314, 0.0314, 0.0314,  ..., 0.0039, 0.0039, 0.0039],
             [0.0353, 0.035

### Training Loop

The cells below contains code to score multiple image text pairs at a given time.

**Training Loop Dev Cell**

In [ ]:
model.train()
losses = []
logit_list = []
targets = []
optim.zero_grad()
for b in batch:
    
    
    
    try:
        infer_dict = model.infer(b)
        logits = model.ref_classifier(infer_dict['cls_feats'])
        logit_list.append(logits.reshape(1,-1))

        obj_ids = b['obj_ids']
        ann_id = b['ann_id']
        target = torch.where(obj_ids==ann_id)[0]
        targets.append(target)
#         target = torch.tensor([obj_ids.index(ann_id)])
    # Adjust learning weights
        
    except RuntimeError:
        print(f'RuntimeError')
loss = loss_fn(logits.reshape(1,-1),target)
losses.append(loss.item())
loss.backward()
optim.step()

In [14]:
model.current_tasks.append('ref')

In [15]:
model(batch)

{'ref_loss': tensor(3.8367, grad_fn=<NllLossBackward>),
 'ref_logits': tensor([[ 1.5193e-01,  3.1318e-01,  3.3413e-01,  4.4152e-01,  4.3146e-01,
           4.4763e-01,  4.5291e-01,  4.3812e-01,  4.3470e-01,  3.5281e-01,
           4.3471e-01,  4.5762e-01,  4.2245e-01,  4.3662e-01,  4.3992e-01,
           4.5509e-01,  1.9790e-01,  4.3491e-01,  4.6142e-01,  4.5360e-01,
           4.3987e-01,  4.4540e-01,  4.3328e-01,  4.4850e-01,  4.3679e-01,
           4.2433e-01,  4.5913e-01,  4.6609e-01,  4.3783e-01,  2.9040e-01,
           4.5130e-01,  4.0505e-01,  2.9442e-01, -4.8169e-02,  1.6322e-01,
          -2.5755e-02,  2.1794e-01, -4.5722e-02, -3.1720e-02,  2.0833e-01,
           3.2046e-02,  4.1486e-02],
         [ 1.3585e-01,  5.2065e-02,  1.9085e-02,  3.8182e-01,  3.3281e-01,
           3.3524e-01,  4.3963e-01,  4.3184e-01,  4.2033e-01,  4.1837e-01,
           3.4357e-01,  4.2899e-01,  3.2228e-01,  3.3480e-01,  3.4037e-01,
           3.5976e-01,  1.9121e-01,  4.7114e-01,  3.3981e-01,  4.265

#### Full Training Loop

In [ ]:
# Create loop for ref res
train_ids = refer.getRefIds(split='train')
text_labels = [[-100 for i in range(40)]]
# train_ids = train_ids[:5]

losses = []
model.train()
for ref_id in tqdm(train_ids):
    ref = refer.Refs[ref_id]
    img_id = ref['image_id']
    ann_id = ref['ann_id']
    objs = refer.imgToAnns[img_id]
    obj_ids = [obj['id'] for obj in objs]
    
    sub_images = []
    for obj in objs:
        x_a = get_bounded_subimage(refer, img_id, obj['id'], xs=224,ys=224, show=False)
        if x_a is not None:
            sub_images.append(x_a)
    num_sub_images = len(sub_images)
        
    
    for sent in ref['sentences']:
        
        
        text_ids = tokenizer.encode(
            sent['sent'],
            padding="max_length",
            truncation=True,
            max_length=40,
            return_special_tokens_mask=True,
        )
        text_masks = [1 if text_ids[i]>0 else 0 for i,_ in enumerate(text_ids)]
        text_labels = [[-100 for i in range(40)]]

        ids = [text_ids for i in range(num_sub_images)]
        masks = [text_masks for _ in range(num_sub_images)]
        labels = [text_labels for i in range(num_sub_images)]
        optim.zero_grad()

        ### TODO: Put all of the sub images in the infer dict with the coressponding sentence.

        input_dict = {
            'image' : sub_images,
            'text' : sent,
            'text_ids' : torch.tensor(ids),
            'text_labels' : torch.tensor(labels),
            'text_masks' : torch.tensor(masks)
        }
        
        infer_dict = model.infer(input_dict)
        logits = model.ref_classifier(infer_dict['cls_feats'])
        
        
        target = torch.tensor([obj_ids.index(ann_id)])
        loss = loss_fn(logits.reshape(1,-1),target)
        losses.append(loss.item())
        loss.backward()

        # Adjust learning weights
        optim.step()

## Test Cells:

### Previous Training Loop

#### Single Example

In [ ]:
ref_id = 0
ref = refer.Refs[ref_id]
img_id = ref['image_id']
ann_id = ref['ann_id']
objs = refer.imgToAnns[img_id]
obj_ids = [obj['id'] for obj in objs]

sub_images = []
for obj in objs:
    x_a = get_bounded_subimage(refer, img_id, obj['id'], xs=224,ys=224, show=False)
    if x_a is not None:
        sub_images.append(x_a)
num_sub_images = len(sub_images)

sent = ref['sentences'][0]
text_ids = tokenizer.encode(
            sent['sent'],
            padding="max_length",
            truncation=True,
            max_length=40,
            return_special_tokens_mask=True,
)
text_masks = [1 if text_ids[i]>0 else 0 for i,_ in enumerate(text_ids)]
text_labels = [[-100 for i in range(40)]]

ids = [text_ids for i in range(num_sub_images)]
masks = [text_masks for _ in range(num_sub_images)]
labels = [text_labels for i in range(num_sub_images)]

input_dict = {
    'image' : sub_images,
    'text' : sent,
    'text_ids' : torch.tensor(ids),
    'text_labels' : torch.tensor(labels),
    'text_masks' : torch.tensor(masks)
}
infer_dict = model.infer(input_dict)
model.ref_classifier(infer_dict['cls_feats'])

#### Full Loop

In [ ]:
# Create loop for ref res
train_ids = refer.getRefIds(split='train')
text_labels = [[-100 for i in range(40)]]
# train_ids = train_ids[:5]

losses = []
model.train()
for ref_id in tqdm(train_ids):
    ref = refer.Refs[ref_id]
    img_id = ref['image_id']
    ann_id = ref['ann_id']
    objs = refer.imgToAnns[img_id]
    obj_ids = [obj['id'] for obj in objs]
    
    sub_images = []
    for obj in objs:
        x_a = get_bounded_subimage(refer, img_id, obj['id'], xs=224,ys=224, show=False)
        if x_a is not None:
            sub_images.append(x_a)
    num_sub_images = len(sub_images)
        
    
    for sent in ref['sentences']:
        
        
        text_ids = tokenizer.encode(
            sent['sent'],
            padding="max_length",
            truncation=True,
            max_length=40,
            return_special_tokens_mask=True,
        )
        text_masks = [1 if text_ids[i]>0 else 0 for i,_ in enumerate(text_ids)]
        text_labels = [[-100 for i in range(40)]]

        ids = [text_ids for i in range(num_sub_images)]
        masks = [text_masks for _ in range(num_sub_images)]
        labels = [text_labels for i in range(num_sub_images)]
        optim.zero_grad()

        ### TODO: Put all of the sub images in the infer dict with the coressponding sentence.

        input_dict = {
            'image' : sub_images,
            'text' : sent,
            'text_ids' : torch.tensor(ids),
            'text_labels' : torch.tensor(labels),
            'text_masks' : torch.tensor(masks)
        }
        
        infer_dict = model.infer(input_dict)
        logits = model.ref_classifier(infer_dict['cls_feats'])
        
        
        target = torch.tensor([obj_ids.index(ann_id)])
        loss = loss_fn(logits.reshape(1,-1),target)
        losses.append(loss.item())
        loss.backward()

        # Adjust learning weights
        optim.step()

### Previous Eval Loop

In [ ]:
## Eval Loop
eval_ids = refer.getRefIds(split='val')[:1]
with torch.no_grad():
    gold = []
    for ref_id in tqdm(eval_ids):
        ref = refer.Refs[ref_id]
        img_id = ref['image_id']
        ann_id = ref['ann_id']
        objs = refer.imgToAnns[img_id]
        obj_ids = [obj['id'] for obj in objs]
    
        sub_images = []
        for obj in objs:
            x_a = get_bounded_subimage(refer, img_id, obj['id'], xs=224,ys=224, show=False)
            if x_a is not None:
                sub_images.append(x_a)
        num_sub_images = len(sub_images)
        for sent in ref['sentences']:
            scores = []
            text_ids = tokenizer.encode(
                sent['sent'],
                padding="max_length",
                truncation=True,
                max_length=40,
                return_special_tokens_mask=True,
            )
            text_masks = torch.tensor([1 if text_ids[i]>0 else 0 for i,_ in enumerate(text_ids)]).reshape(1,-1)
    
            for sub_image in sub_images:
                # assert not torch.isnan(x_a).any()
                input_dict = {
                    'image' : [sub_image],
                    'text' : sent,
                    'text_ids' : torch.tensor(text_ids).reshape(1,-1),
                    'text_labels' : text_labels,
                    'text_masks' : text_masks
                }
                infer_dict = model.infer(input_dict)
                score = model.ref_classifier(infer_dict['cls_feats'])
                scores.append(score)

            pred_index = np.argmax(scores)
            pred_id = objs[pred_index]['id']
            target = torch.tensor([obj_ids.index(ann_id)])
            scores = torch.cat(scores)
            if pred_id == ann_id:
                gold.append(1)
            else:
                gold.append(0)